# PCA / t-SNE / UMAP Embedding Visualisations

Reads pre-computed parquet files and produces:
- Per-layer PNG plots (2D by modality & scene category) — **per pair**
- Grid PNGs — **per pair**
- Interactive 3D HTML with **modality-pair dropdown + layer slider** (PCA · t-SNE · UMAP)

Outputs → `outputs/<run_id>/<dataset>/`

| Method | 2D PNG | 3D HTML (pair dropdown + layer slider) |
|--------|--------|----------------------------------------|
| PCA   | ✅ per pair | ✅ all pairs in one figure |
| t-SNE | ✅ per pair | ✅ all pairs in one figure |
| UMAP  | ✅ per pair | ✅ all pairs in one figure |


## 0 · Configuration — **edit here**

In [ ]:
from pathlib import Path

# ── USER CONFIG ───────────────────────────────────────────────────────────────
RUN_ID  = "rq1_full_pipeline_hypersim_100_scenes_4000_samples_join_null_A-20260518-225144"
DATASET = "hypersim"          # "hypersim" or "diode"
SUBDIR  = "embeddings_umap"   # subfolder under results/embeddings/<run_id>/

# Pairs to load — adjust to whichever pairs you have computed
PAIRS = [
    "rgb-depth",
    "depth-normals",
    "rgb-normals",
]

REPO_ROOT     = Path("..").resolve()
OUT_ROOT      = REPO_ROOT / "outputs" / RUN_ID / DATASET
OUT_ROOT.mkdir(parents=True, exist_ok=True)

LAYERS_FILTER = None   # e.g. ["layer_00", "layer_05", "layer_11"] or None = all
DPI           = 150
GRID_NCOLS    = 4
MAX_FRAMES    = 5_000
RANDOM_SEED   = 42

print(f"Repo root  : {REPO_ROOT}")
print(f"Pairs      : {PAIRS}")
print(f"Output root: {OUT_ROOT}")

## 1 · Imports & helpers

In [ ]:
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, IFrame

try:
    import plotly.graph_objects as go
    import plotly.offline as pyo
    pyo.init_notebook_mode(connected=True)
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    warnings.warn("plotly not installed — 3D HTML skipped.")

MOD_PALETTE = {"rgb": "#E07B39", "depth": "#4A90D9", "normals": "#57A85A"}

def mod_color(m): return MOD_PALETTE.get(m, "#888888")

def category_palette(cats):
    cmap = plt.get_cmap("tab20")
    cats = sorted(cats)
    return {c: cmap(i/max(len(cats)-1,1)) for i,c in enumerate(cats)}

def cat_hex_palette(cats):
    cmap = plt.get_cmap("tab20")
    cats = sorted(cats)
    def h(rgba): return "#{:02x}{:02x}{:02x}".format(*[int(v*255) for v in rgba[:3]])
    return {c: h(cmap(i/max(len(cats)-1,1))) for i,c in enumerate(cats)}

print("Imports OK")

## 2 · Load data — all pairs

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

frames_by_pair   = {}   # {pair: {layer: df}}
pca_vars_by_pair = {}   # {pair: {layer: ndarray|None}}

for pair in PAIRS:
    emb_dir = REPO_ROOT / "results" / "embeddings" / RUN_ID / SUBDIR / pair
    if not emb_dir.exists():
        print(f"  [SKIP] {pair} — directory not found: {emb_dir}")
        continue
    avail = sorted(p.name for p in emb_dir.iterdir()
                   if p.is_dir() and p.name.startswith("layer_"))
    layers_to_use = LAYERS_FILTER or avail
    frames_by_pair[pair]   = {}
    pca_vars_by_pair[pair] = {}
    for layer in layers_to_use:
        fp = emb_dir / layer / "frame.parquet"
        if not fp.exists(): continue
        df = pd.read_parquet(fp)
        if MAX_FRAMES and len(df) > MAX_FRAMES:
            idx = rng.choice(len(df), MAX_FRAMES, replace=False)
            df  = df.iloc[idx].reset_index(drop=True)
        frames_by_pair[pair][layer] = df
        vp = emb_dir / layer / "pca_variance.csv"
        pca_vars_by_pair[pair][layer] = (
            pd.read_csv(vp)["explained_variance_ratio"].values[:3]
            if vp.exists() else None
        )
    n = len(frames_by_pair[pair])
    print(f"  {pair:50s}: {n} layers loaded")

PAIRS_LOADED = [p for p in PAIRS if frames_by_pair.get(p)]
LAYERS = sorted({l for fp in frames_by_pair.values() for l in fp})
# backward-compat aliases for the first pair
PAIR      = PAIRS_LOADED[0] if PAIRS_LOADED else PAIRS[0]
frames    = frames_by_pair.get(PAIR, {})
pca_vars  = pca_vars_by_pair.get(PAIR, {})

print(f"\nPairs loaded : {PAIRS_LOADED}")
print(f"Layers       : {LAYERS}")
if frames:
    sample = next(iter(frames.values()))
    print(f"Columns      : {list(sample.columns)}")

### 2b · Embedding availability

In [ ]:
if frames:
    cols = set(next(iter(frames.values())).columns)
    checks = [
        ("PCA  2D", {"pca_x","pca_y"}),       ("PCA  3D", {"pca3_x","pca3_y","pca3_z"}),
        ("t-SNE 2D",{"tsne_x","tsne_y"}),      ("t-SNE 3D",{"tsne3_x","tsne3_y","tsne3_z"}),
        ("UMAP 2D", {"umap_x","umap_y"}),      ("UMAP 3D", {"umap3_x","umap3_y","umap3_z"}),
    ]
    HAS_PCA2,HAS_PCA3,HAS_TSNE2,HAS_TSNE3,HAS_UMAP2,HAS_UMAP3 = [
        c.issubset(cols) for _,c in checks]
    for name,c in checks:
        print(f"  {name}: {'✅' if c.issubset(cols) else '❌  (rerun compute_embeddings.py)'}")

### 2c · Global palettes (consistent across pairs & layers)

In [ ]:
all_cats = sorted({
    c for fp in frames_by_pair.values()
    for df in fp.values()
    for c in df["scene_type"].unique()
})
cat_pal = category_palette(all_cats)
cat_hex = cat_hex_palette(all_cats)
print(f"Scene categories ({len(all_cats)}): {all_cats}")

## 3 · Shared helpers — 3D & 2D

In [ ]:
def _has_cols(df, *cols): return set(cols).issubset(df.columns)
def _has_pca3(df):  return _has_cols(df,"pca3_x","pca3_y","pca3_z")
def _has_tsne3(df): return _has_cols(df,"tsne3_x","tsne3_y","tsne3_z")
def _has_umap3(df): return _has_cols(df,"umap3_x","umap3_y","umap3_z")

def _hex(cmap, val): return cmap.get(val,"#888888") if cmap else "#888888"

# Default camera — trace draw order is tuned for this viewpoint.
_DEFAULT_CAMERA_EYE = dict(x=1.4, y=1.4, z=0.8)

def _3d_axis(title="", axis_range=None):
    """3D axis with optional fixed range — prevents rescale on legend/slider."""
    ax = dict(title=dict(text=title, font=dict(size=11)),
              showticklabels=False, showgrid=True,
              gridcolor="rgba(180,180,180,0.6)", gridwidth=1,
              zeroline=True, zerolinecolor="rgba(150,150,150,0.8)",
              zerolinewidth=1.5, backgroundcolor="rgb(240,242,248)",
              showspikes=False,
              autorange=False)  # fixedrange not supported on 3D scene axes
    if axis_range is not None:
        ax["range"] = list(axis_range)
    return ax


def _global_3d_ranges(frames_aligned, valid_by_pair, dims, pad=0.05):
    """Global [lo, hi] per axis across all pairs/layers (equal span for cube)."""
    xcol, ycol, zcol = dims
    xs, ys, zs = [], [], []
    for pair, layers in valid_by_pair.items():
        for layer in layers:
            df = frames_aligned[pair][layer]
            xs.append(df[xcol].to_numpy())
            ys.append(df[ycol].to_numpy())
            zs.append(df[zcol].to_numpy())
    xs = np.concatenate(xs); ys = np.concatenate(ys); zs = np.concatenate(zs)

    def lims(arr):
        lo, hi = float(np.min(arr)), float(np.max(arr))
        span = hi - lo
        p = span * pad if span > 0 else 1.0
        return lo - p, hi + p

    rx, ry, rz = lims(xs), lims(ys), lims(zs)
    cx = (rx[0] + rx[1]) / 2; cy = (ry[0] + ry[1]) / 2; cz = (rz[0] + rz[1]) / 2
    half = max((rx[1]-rx[0])/2, (ry[1]-ry[0])/2, (rz[1]-rz[0])/2)
    return {
        "x": [cx - half, cx + half],
        "y": [cy - half, cy + half],
        "z": [cz - half, cz + half],
    }


def _scene_axis_patch(ax_labels, axis_ranges):
    """Layout patch that keeps axis box + grid fixed (slider, legend, dropdown)."""
    return {
        "scene.xaxis": _3d_axis(ax_labels[0], axis_ranges["x"]),
        "scene.yaxis": _3d_axis(ax_labels[1], axis_ranges["y"]),
        "scene.zaxis": _3d_axis(ax_labels[2], axis_ranges["z"]),
        "scene.aspectmode": "cube",
        "scene.uirevision": "locked",
    }

def _base_layout(title, width=900, height=660):
    return dict(template="plotly_white",
                title=dict(text=title, font=dict(size=13), x=0.5, xanchor="center"),
                width=width, height=height,
                margin=dict(l=0,r=0,t=55,b=0),
                legend=dict(itemsizing="constant", font=dict(size=11),
                            bgcolor="rgba(255,255,255,0.85)",
                            bordercolor="rgba(0,0,0,0.15)", borderwidth=1),
                scene=dict(xaxis=_3d_axis(), yaxis=_3d_axis(), zaxis=_3d_axis(),
                           aspectmode="cube", uirevision="locked",
                           camera=dict(eye=_DEFAULT_CAMERA_EYE,
                                       up=dict(x=0,y=0,z=1)),
                           bgcolor="rgb(255,255,255)"),
                paper_bgcolor="white", plot_bgcolor="white")

def _hover_cols(df):
    cols = [c for c in ["scene_type","scene_short","sample_key"] if c in df.columns]
    if not cols: return None
    return df[cols].apply(lambda r:" | ".join(str(r[c]) for c in cols),axis=1)


def _trace_draw_order(df, x, y, z, color_col, eye=_DEFAULT_CAMERA_EYE):
    """Sort colour groups back-to-front for the default camera (far drawn first)."""
    eye_vec = np.array([eye["x"], eye["y"], eye["z"]], dtype=float)
    eye_vec /= np.linalg.norm(eye_vec) or 1.0

    def depth_key(val):
        sub = df.loc[df[color_col] == val, [x, y, z]]
        if sub.empty:
            return 0.0
        c = sub.mean().to_numpy()
        return float(np.dot(c, eye_vec))  # larger = closer to camera → draw later

    # ascending depth: far groups first, near groups on top
    return sorted(df[color_col].unique(), key=depth_key)


def _make_traces(df, x, y, z, color_col, color_map, opacity=0.72):
    """One Scatter3d trace per colour; order avoids wrong overlap with transparency."""
    hover_base = _hover_cols(df)
    traces = []
    for val in _trace_draw_order(df, x, y, z, color_col):
        m = df[color_col] == val
        sub = df[m]
        hover_text = ((hover_base[m].astype(str) + " | " + sub["modality"].astype(str))
                      if hover_base is not None else sub["modality"].astype(str))
        traces.append(go.Scatter3d(
            x=sub[x], y=sub[y], z=sub[z], mode="markers",
            name=str(val),
            marker=dict(size=2.5, color=_hex(color_map, val),
                        opacity=opacity, line=dict(width=0)),
            text=hover_text,
            hovertemplate="%{text}<br>1: %{x:.2f}  2: %{y:.2f}  3: %{z:.2f}"
                          "<extra>" + str(val) + "</extra>",
        ))
    return traces

def make_pca3d_traces(df, color_col="modality", color_map=None, opacity=0.55):
    return _make_traces(df,"pca3_x","pca3_y","pca3_z",color_col,color_map,opacity)
def make_tsne3d_traces(df, color_col="modality", color_map=None, opacity=0.55):
    return _make_traces(df,"tsne3_x","tsne3_y","tsne3_z",color_col,color_map,opacity)
def make_umap3d_traces(df, color_col="modality", color_map=None, opacity=0.55):
    return _make_traces(df,"umap3_x","umap3_y","umap3_z",color_col,color_map,opacity)

def _var_labels(pca_vars_dict, pair, layer, method):
    if method == "pca":
        var = (pca_vars_dict or {}).get(pair, {}).get(layer)
        if var is not None and len(var) >= 3:
            return [f"PC{i+1} ({var[i]*100:.1f}%)" for i in range(3)]
        return ["PC1","PC2","PC3"]
    elif method == "tsne": return ["dim 1","dim 2","dim 3"]
    else: return ["UMAP 1","UMAP 2","UMAP 3"]


# ── Sign alignment across layers ─────────────────────────────────────────────
# PCA / t-SNE / UMAP axes are only defined up to independent sign flips per axis.
# Without alignment, the same structure can appear mirrored between layers.
# We pick sign flips (per axis) that minimise distance between modality centroids
# vs layer_00 — same approach as analysis_notebook.ipynb.

import itertools

def align_signs(frames, layers, dims, group="modality"):
    """Return {layer: df} with per-axis sign flips chosen so modality centroids
    stay consistent with the first available layer."""
    dims = tuple(dims)  # e.g. ("pca_x", "pca_y") or 3D coords
    layers_ok = [l for l in layers
                 if l in frames and all(d in frames[l].columns for d in dims)]
    if len(layers_ok) < 2 or group not in frames[layers_ok[0]].columns:
        return dict(frames)

    def centroids(df):
        return {m: df.loc[df[group] == m, list(dims)].mean().values
                for m in df[group].unique()}

    ref_df = frames[layers_ok[0]]
    ref_c  = centroids(ref_df)
    out    = {layers_ok[0]: ref_df.copy()}

    sign_grid = list(itertools.product([1, -1], repeat=len(dims)))
    for layer in layers_ok[1:]:
        df = frames[layer].copy()
        best_cost, best_signs = np.inf, sign_grid[0]
        for signs in sign_grid:
            cand = df.copy()
            for d, s in zip(dims, signs):
                cand[d] = df[d].values * s
            c = centroids(cand)
            cost = sum(np.sum((ref_c[m] - c[m]) ** 2)
                       for m in ref_c if m in c)
            if cost < best_cost:
                best_cost, best_signs = cost, signs
        for d, s in zip(dims, best_signs):
            df[d] = df[d].values * s
        out[layer] = df

    for l in layers:
        if l in frames and l not in out:
            out[l] = frames[l]
    return out


print("3D helpers loaded.")

In [ ]:
# ── Multi-pair 3D figure: dropdown (pair) + slider (layer) ───────────────────

# PAIR_LABELS: short readable names for the dropdown
PAIR_LABELS = {
    "rgb-depth":                                    "RGB ↔ Depth",
    "depth-normals":                                "Depth ↔ Normals",
    "rgb-normals":                                  "RGB ↔ Normals",
    "rgb-rgb_joint_with_depth":                     "RGB | +D",
    "rgb-rgb_joint_with_normals":                   "RGB | +N",
    "depth-depth_joint_with_rgb":                   "Depth | +RGB",
    "depth-depth_joint_with_normals":               "Depth | +N",
    "normals-normals_joint_with_rgb":               "Normals | +RGB",
    "normals-normals_joint_with_depth":             "Normals | +D",
    "rgb_joint_with_depth-depth_joint_with_rgb":    "RGB|+D ↔ D|+RGB",
    "rgb_joint_with_normals-normals_joint_with_rgb":"RGB|+N ↔ N|+RGB",
    "depth_joint_with_normals-normals_joint_with_depth":"D|+N ↔ N|+D",
}

def _pair_label(pair): return PAIR_LABELS.get(pair, pair)


# Map (trace builder → dims) so we can sign-align per-pair before plotting.
_TRACE_DIMS = {
    "make_pca3d_traces":  ("pca3_x",  "pca3_y",  "pca3_z"),
    "make_tsne3d_traces": ("tsne3_x", "tsne3_y", "tsne3_z"),
    "make_umap3d_traces": ("umap3_x", "umap3_y", "umap3_z"),
}


def make_multipair_3d(frames_by_pair, layers, trace_fn, has_fn,
                       color_col, color_map, title, method,
                       pca_vars_by_pair=None, width=920, height=720):
    """Single 3D figure covering ALL loaded pairs.

    Controls
    --------
    * Dropdown   → choose modality pair
    * Slider     → choose layer within the current pair

    Selecting a pair via the dropdown also reconfigures the slider so its steps
    match that pair\'s available layers (and resets the slider to layer 0).
    Camera AND axis ranges/grid stay fixed (global limits, autorange=False).
    """
    if not HAS_PLOTLY: return None

    dims = _TRACE_DIMS.get(trace_fn.__name__, None)

    # collect valid layers per pair, applying sign alignment
    valid_by_pair = {}      # {pair: [layer, ...]}
    frames_aligned = {}     # {pair: {layer: df}}
    for p, fp in frames_by_pair.items():
        if dims is not None:
            aligned = align_signs(fp, layers, dims)
        else:
            aligned = dict(fp)
        ok = [l for l in layers if l in aligned and has_fn(aligned[l])]
        if ok:
            valid_by_pair[p]  = ok
            frames_aligned[p] = aligned

    if not valid_by_pair:
        print(f"[SKIP] no valid (pair,layer) for {method} 3D"); return None

    pairs_ok = list(valid_by_pair.keys())

    # Fixed axis box across ALL pairs/layers (no rescale on slider or legend)
    axis_ranges = _global_3d_ranges(frames_aligned, valid_by_pair, dims) if dims else None

    # ── build all traces, track offsets per (pair, layer) ────────────────────
    all_traces = []
    offsets    = {}     # {(pair, layer): (start, count)}
    for p in pairs_ok:
        for l in valid_by_pair[p]:
            start  = len(all_traces)
            df     = frames_aligned[p][l]
            traces = trace_fn(df, color_col=color_col, color_map=color_map)
            offsets[(p, l)] = (start, len(traces))
            for tr in traces: tr.visible = False
            all_traces.extend(traces)

    first_pair  = pairs_ok[0]
    first_layer = valid_by_pair[first_pair][0]
    s, c = offsets[(first_pair, first_layer)]
    for k in range(c): all_traces[s+k].visible = True

    def visibility_for(pair, layer):
        vis = [False] * len(all_traces)
        s, c = offsets[(pair, layer)]
        for k in range(c): vis[s+k] = True
        return vis

    pv = pca_vars_by_pair or {}

    def slider_for(pair):
        """Slider config whose steps walk over the given pair\'s layers."""
        steps = []
        for layer in valid_by_pair[pair]:
            ax = _var_labels(pv, pair, layer, method)
            steps.append(dict(
                label=layer.replace("layer_", "L"),
                method="update",
                args=[
                    {"visible": visibility_for(pair, layer)},
                    dict(
                        title=dict(text=f"{title} — {_pair_label(pair)} · {layer}"),
                        **(_scene_axis_patch(ax, axis_ranges) if axis_ranges else {
                            "scene.xaxis.title.text": ax[0],
                            "scene.yaxis.title.text": ax[1],
                            "scene.zaxis.title.text": ax[2],
                            "scene.uirevision": "locked",
                        }),
                    ),
                ],
            ))
        return dict(
            active=0,
            currentvalue=dict(prefix="Layer: ", font=dict(size=11)),
            pad=dict(t=30, b=10),
            len=0.85, x=0.075, y=0,
            steps=steps,
        )

    # ── dropdown buttons: one per pair (rebuilds the slider on click) ────────
    buttons = []
    for pair in pairs_ok:
        first_l = valid_by_pair[pair][0]
        ax = _var_labels(pv, pair, first_l, method)
        buttons.append(dict(
            label=_pair_label(pair),
            method="update",
            args=[
                {"visible": visibility_for(pair, first_l)},
                dict(
                    title=dict(text=f"{title} — {_pair_label(pair)} · {first_l}"),
                    sliders=[slider_for(pair)],
                    **(_scene_axis_patch(ax, axis_ranges) if axis_ranges else {
                        "scene.xaxis.title.text": ax[0],
                        "scene.yaxis.title.text": ax[1],
                        "scene.zaxis.title.text": ax[2],
                        "scene.uirevision": "locked",
                    }),
                ),
            ],
        ))

    # ── layout ───────────────────────────────────────────────────────────────
    ax0 = _var_labels(pv, first_pair, first_layer, method)
    layout = _base_layout(
        f"{title} — {_pair_label(first_pair)} · {first_layer}",
        width=width, height=height,
    )
    if axis_ranges:
        layout["scene"].update(
            xaxis=_3d_axis(ax0[0], axis_ranges["x"]),
            yaxis=_3d_axis(ax0[1], axis_ranges["y"]),
            zaxis=_3d_axis(ax0[2], axis_ranges["z"]),
            aspectmode="cube", uirevision="locked",
        )
    else:
        layout["scene"].update(
            xaxis=_3d_axis(ax0[0]), yaxis=_3d_axis(ax0[1]), zaxis=_3d_axis(ax0[2]),
            uirevision="locked",
        )
    layout["updatemenus"] = [dict(
        type="dropdown", direction="down",
        x=0.0, xanchor="left",
        y=1.13, yanchor="top",
        showactive=True, bgcolor="white",
        bordercolor="rgba(0,0,0,0.2)",
        font=dict(size=11),
        buttons=buttons,
    )]
    layout["sliders"] = [slider_for(first_pair)]
    layout["annotations"] = [dict(
        text="Modality pair:", x=0.0, y=1.17,
        xref="paper", yref="paper",
        showarrow=False, font=dict(size=11),
    )]
    layout["margin"]["t"] = 90
    layout["margin"]["b"] = 90   # extra room for the slider

    fig = go.Figure(data=all_traces)
    fig.update_layout(layout)
    n_combos = sum(len(v) for v in valid_by_pair.values())
    print(f"  Built: {len(pairs_ok)} pairs · {n_combos} (pair·layer) groups · "
          f"{len(all_traces)} traces · dropdown + slider")
    return fig


def save_multipair_3d(frames_by_pair, layers, trace_fn, has_fn,
                       color_col, color_map, out_path, title, method,
                       pca_vars_by_pair=None, include_plotlyjs="cdn"):
    fig = make_multipair_3d(frames_by_pair, layers, trace_fn, has_fn,
                             color_col, color_map, title, method,
                             pca_vars_by_pair=pca_vars_by_pair)
    if fig is None: return None
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(out_path), include_plotlyjs=include_plotlyjs)
    print(f"  Saved → {out_path}")
    return fig

print("Multi-pair 3D helpers loaded.")


In [ ]:
# Verify interactive controls (run after cell 13)
import inspect
src = inspect.getsource(make_multipair_3d)
assert "slider_for" in src, "make_multipair_3d missing slider_for — re-run cell 13"
assert "Pair · Layer" not in src, "old combined dropdown still present"
print("✓ 3D controls: pair dropdown + layer slider")
print("✓ align_signs:", "centroids" in inspect.getsource(align_signs))


In [ ]:
# ── 2D PNG helpers (unchanged, loop over pairs externally) ───────────────────

def _2d_lims(frames, layers, x, y, pad=0.05):
    ax = np.concatenate([frames[l][x].values for l in layers if x in frames[l].columns])
    ay = np.concatenate([frames[l][y].values for l in layers if y in frames[l].columns])
    xp=(ax.max()-ax.min())*pad; yp=(ay.max()-ay.min())*pad
    return (ax.min()-xp,ax.max()+xp),(ay.min()-yp,ay.max()+yp)

def _scatter2d(df,x,y,col,cmap,ax,layer,xl="",yl="",xlim=None,ylim=None):
    for val in sorted(df[col].unique()):
        m = df[col]==val
        c = cmap.get(val,"#888") if cmap else None
        ax.scatter(df.loc[m,x],df.loc[m,y],c=[c],label=val,alpha=0.45,s=6,rasterized=True)
    ax.set_xlabel(xl,fontsize=8); ax.set_ylabel(yl,fontsize=8)
    ax.set_title(layer,fontsize=9); ax.grid(alpha=0.2)
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)

def run_2d_section(frames, layers, x, y, has_fn, col, cmap,
                   method_label, out_dir, suptitle, xl="",yl="",leg_title=""):
    layers_ok=[l for l in layers if l in frames and has_fn(frames[l])]
    if not layers_ok: print(f"[SKIP] {method_label} 2D for {x}"); return
    # Sign-align axes across layers so the orientation is stable from layer to layer
    frames = align_signs(frames, layers_ok, (x, y))
    xlim,ylim = _2d_lims(frames,layers_ok,x,y)
    per = Path(out_dir)/"per_layer"; per.mkdir(parents=True,exist_ok=True)
    for layer in layers_ok:
        fig2,ax2=plt.subplots(figsize=(7 if col=="scene_type" else 6,5))
        _scatter2d(frames[layer],x,y,col,cmap,ax2,layer,xl,yl,xlim,ylim)
        ax2.set_title(f"{method_label} 2D — {col}\n{layer}",fontsize=10)
        if col=="scene_type":
            ax2.legend(title=leg_title or col,markerscale=3,fontsize=7,
                       bbox_to_anchor=(1.05,1),loc="upper left")
        else:
            ax2.legend(title=leg_title or col,markerscale=3,fontsize=8)
        fig2.tight_layout()
        fig2.savefig(per/f"{layer}.png",dpi=DPI,bbox_inches="tight")
        plt.close(fig2)
    ncols=GRID_NCOLS; nrows=int(np.ceil(len(layers_ok)/ncols))
    fig3,axes=plt.subplots(nrows,ncols,figsize=(ncols*4,nrows*3.5),sharex=True,sharey=True)
    axes=axes.flatten()
    for i,layer in enumerate(layers_ok):
        _scatter2d(frames[layer],x,y,col,cmap,axes[i],layer,xl,yl,xlim,ylim)
    for j in range(i+1,len(axes)): axes[j].set_visible(False)
    all_vals=sorted({v for df in frames.values() for v in df[col].unique()})
    handles=[mpatches.Patch(color=cmap.get(v,"#888") if cmap else "#888",label=v) for v in all_vals]
    if col=="scene_type":
        fig3.legend(handles=handles,title=leg_title or col,loc="lower center",
                    bbox_to_anchor=(0.5,0.0),fontsize=7,ncol=6,frameon=True)
        fig3.subplots_adjust(bottom=0.12)
    else:
        fig3.legend(handles=handles,title=leg_title or col,loc="lower right",fontsize=8)
    fig3.suptitle(suptitle,fontsize=12); fig3.tight_layout()
    gp=Path(out_dir)/"grid_all_layers.png"; gp.parent.mkdir(parents=True,exist_ok=True)
    fig3.savefig(gp,dpi=DPI,bbox_inches="tight")
    plt.show(); print(f"Grid → {gp}")
    print(f"Per-layer → {per}")

def run_3d_grid_png(frames,layers,x,y,has_fn,col,cmap,out_path,suptitle,xl="",yl=""):
    # Sign-align (x, y) across layers; z is unused in this 2D grid view.
    frames = align_signs(frames, layers, (x, y))
    ncols=GRID_NCOLS; nrows=int(np.ceil(len(layers)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*4,nrows*3.5))
    axes=axes.flatten()
    for i,layer in enumerate(layers):
        df,ax=frames.get(layer),axes[i]
        if df is None or not has_fn(df):
            ax.text(0.5,0.5,"missing",ha="center",va="center"); ax.axis("off"); continue
        for val in sorted(df[col].unique()):
            m=df[col]==val; c=cmap.get(val,"#888") if cmap else None
            ax.scatter(df.loc[m,x],df.loc[m,y],c=[c],label=val,alpha=0.4,s=4,rasterized=True)
        ax.set_xlabel(xl,fontsize=7); ax.set_ylabel(yl,fontsize=7)
        ax.set_title(layer,fontsize=9); ax.grid(alpha=0.2)
    for j in range(i+1,len(axes)): axes[j].set_visible(False)
    all_vals=sorted({v for df in frames.values() for v in df[col].unique()})
    handles=[mpatches.Patch(color=cmap.get(v,"#888") if cmap else "#888",label=v) for v in all_vals]
    fig.legend(handles=handles,title=col,loc="lower right",fontsize=7,ncol=2)
    fig.suptitle(suptitle,fontsize=12); fig.tight_layout()
    out_path=Path(out_path); out_path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(out_path,dpi=DPI,bbox_inches="tight")
    plt.show(); print(f"Grid PNG → {out_path}")

print("2D helpers loaded.")

---
## 4 · PCA 2D — per pair

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    print(f"\n── {pair} ──")
    run_2d_section(fp, LAYERS, "pca_x","pca_y",
        lambda df: _has_cols(df,"pca_x","pca_y"),
        "modality", MOD_PALETTE,
        "PCA", OUT_ROOT/pair/"pca2d_modality",
        f"PCA 2D — modality | {_pair_label(pair)} | {DATASET}",
        xl="PC1", yl="PC2")

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    print(f"\n── {pair} ──")
    run_2d_section(fp, LAYERS, "pca_x","pca_y",
        lambda df: _has_cols(df,"pca_x","pca_y"),
        "scene_type", cat_pal,
        "PCA", OUT_ROOT/pair/"pca2d_category",
        f"PCA 2D — scene category | {_pair_label(pair)} | {DATASET}",
        xl="PC1", yl="PC2", leg_title="Scene category")

---
## 5 · PCA 3D — all pairs in one interactive figure

Use the **modality-pair dropdown** (top) to switch pairs, and the **layer slider** (bottom) to step through layers L00–L11.

The camera stays fixed across all selections (`uirevision=locked`) — rotate once, then explore pairs and layers.

2D plots apply the same sign alignment across layers so PCA / t-SNE / UMAP orientations stay comparable.


### 5a · PCA 3D — coloured by **modality**

In [ ]:
fig_pca_mod = save_multipair_3d(
    frames_by_pair, LAYERS, make_pca3d_traces, _has_pca3,
    color_col="modality", color_map=MOD_PALETTE,
    out_path=OUT_ROOT / "pca3d_modality_multipair.html",
    title="PCA 3D — modality", method="pca",
    pca_vars_by_pair=pca_vars_by_pair,
)
if fig_pca_mod: fig_pca_mod.show()

### 5b · PCA 3D — coloured by **scene category**

In [ ]:
fig_pca_cat = save_multipair_3d(
    frames_by_pair, LAYERS, make_pca3d_traces, _has_pca3,
    color_col="scene_type", color_map=cat_hex,
    out_path=OUT_ROOT / "pca3d_category_multipair.html",
    title="PCA 3D — scene category", method="pca",
    pca_vars_by_pair=pca_vars_by_pair,
)
if fig_pca_cat: fig_pca_cat.show()

### 5c · PCA 3D — grid PNG (static overview, per pair)

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    for col, cmap, suffix in [("modality",MOD_PALETTE,"modality"),
                               ("scene_type",cat_pal,"category")]:
        run_3d_grid_png(fp, LAYERS, "pca3_x","pca3_y", _has_pca3,
            col, cmap,
            OUT_ROOT/pair/f"pca3d_{suffix}"/"grid_all_layers.png",
            f"PCA 3D (PC1 vs PC2) — {suffix} | {_pair_label(pair)} | {DATASET}",
            xl="PC1", yl="PC2")

---
## 6 · t-SNE 2D — per pair

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    print(f"\n── {pair} ──")
    run_2d_section(fp, LAYERS, "tsne_x","tsne_y",
        lambda df: _has_cols(df,"tsne_x","tsne_y"),
        "modality", MOD_PALETTE,
        "t-SNE", OUT_ROOT/pair/"tsne2d_modality",
        f"t-SNE 2D — modality | {_pair_label(pair)} | {DATASET}",
        xl="dim 1", yl="dim 2")

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    print(f"\n── {pair} ──")
    run_2d_section(fp, LAYERS, "tsne_x","tsne_y",
        lambda df: _has_cols(df,"tsne_x","tsne_y"),
        "scene_type", cat_pal,
        "t-SNE", OUT_ROOT/pair/"tsne2d_category",
        f"t-SNE 2D — scene category | {_pair_label(pair)} | {DATASET}",
        xl="dim 1", yl="dim 2", leg_title="Scene category")

---
## 7 · t-SNE 3D — all pairs in one interactive figure

### 7a · t-SNE 3D — coloured by **modality**

In [ ]:
fig_tsne_mod = save_multipair_3d(
    frames_by_pair, LAYERS, make_tsne3d_traces, _has_tsne3,
    color_col="modality", color_map=MOD_PALETTE,
    out_path=OUT_ROOT / "tsne3d_modality_multipair.html",
    title="t-SNE 3D — modality", method="tsne",
    pca_vars_by_pair=pca_vars_by_pair,
)
if fig_tsne_mod: fig_tsne_mod.show()

### 7b · t-SNE 3D — coloured by **scene category**

In [ ]:
fig_tsne_cat = save_multipair_3d(
    frames_by_pair, LAYERS, make_tsne3d_traces, _has_tsne3,
    color_col="scene_type", color_map=cat_hex,
    out_path=OUT_ROOT / "tsne3d_category_multipair.html",
    title="t-SNE 3D — scene category", method="tsne",
    pca_vars_by_pair=pca_vars_by_pair,
)
if fig_tsne_cat: fig_tsne_cat.show()

### 7c · t-SNE 3D — grid PNG (static overview, per pair)

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    for col,cmap,suffix in [("modality",MOD_PALETTE,"modality"),
                             ("scene_type",cat_pal,"category")]:
        run_3d_grid_png(fp, LAYERS,"tsne3_x","tsne3_y",_has_tsne3,col,cmap,
            OUT_ROOT/pair/f"tsne3d_{suffix}"/"grid_all_layers.png",
            f"t-SNE 3D — {suffix} | {_pair_label(pair)} | {DATASET}",
            xl="dim 1", yl="dim 2")

---
## 8 · UMAP 2D — per pair

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    print(f"\n── {pair} ──")
    run_2d_section(fp, LAYERS, "umap_x","umap_y",
        lambda df: _has_cols(df,"umap_x","umap_y"),
        "modality", MOD_PALETTE,
        "UMAP", OUT_ROOT/pair/"umap2d_modality",
        f"UMAP 2D — modality | {_pair_label(pair)} | {DATASET}",
        xl="UMAP 1", yl="UMAP 2")

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    print(f"\n── {pair} ──")
    run_2d_section(fp, LAYERS, "umap_x","umap_y",
        lambda df: _has_cols(df,"umap_x","umap_y"),
        "scene_type", cat_pal,
        "UMAP", OUT_ROOT/pair/"umap2d_category",
        f"UMAP 2D — scene category | {_pair_label(pair)} | {DATASET}",
        xl="UMAP 1", yl="UMAP 2", leg_title="Scene category")

---
## 9 · UMAP 3D — all pairs in one interactive figure

### 9a · UMAP 3D — coloured by **modality**

In [ ]:
fig_umap_mod = save_multipair_3d(
    frames_by_pair, LAYERS, make_umap3d_traces, _has_umap3,
    color_col="modality", color_map=MOD_PALETTE,
    out_path=OUT_ROOT / "umap3d_modality_multipair.html",
    title="UMAP 3D — modality", method="umap",
    pca_vars_by_pair=pca_vars_by_pair,
)
if fig_umap_mod: fig_umap_mod.show()

### 9b · UMAP 3D — coloured by **scene category**

In [ ]:
fig_umap_cat = save_multipair_3d(
    frames_by_pair, LAYERS, make_umap3d_traces, _has_umap3,
    color_col="scene_type", color_map=cat_hex,
    out_path=OUT_ROOT / "umap3d_category_multipair.html",
    title="UMAP 3D — scene category", method="umap",
    pca_vars_by_pair=pca_vars_by_pair,
)
if fig_umap_cat: fig_umap_cat.show()

### 9c · UMAP 3D — grid PNG (static overview, per pair)

In [ ]:
for pair in PAIRS_LOADED:
    fp = frames_by_pair[pair]
    for col,cmap,suffix in [("modality",MOD_PALETTE,"modality"),
                             ("scene_type",cat_pal,"category")]:
        run_3d_grid_png(fp, LAYERS,"umap3_x","umap3_y",_has_umap3,col,cmap,
            OUT_ROOT/pair/f"umap3d_{suffix}"/"grid_all_layers.png",
            f"UMAP 3D — {suffix} | {_pair_label(pair)} | {DATASET}",
            xl="UMAP 1", yl="UMAP 2")

---
## 10 · PCA cumulative explained variance — per pair

In [ ]:
for pair in PAIRS_LOADED:
    pv = pca_vars_by_pair.get(pair, {})
    layers_ok = [l for l in LAYERS if pv.get(l) is not None]
    if not layers_ok: print(f"[SKIP] {pair}"); continue
    ncols=GRID_NCOLS; nrows=int(np.ceil(len(layers_ok)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*4,nrows*3))
    axes=axes.flatten()
    for i,layer in enumerate(layers_ok):
        var=pv[layer]; ax=axes[i]
        cum=np.cumsum(var)*100
        ax.plot(range(1,len(cum)+1),cum,lw=1.5)
        ax.axhline(90,color="r",ls="--",lw=0.8); ax.axhline(95,color="orange",ls="--",lw=0.8)
        ax.set_title(layer,fontsize=8); ax.set_xlabel("Components",fontsize=7)
        ax.set_ylabel("Cum var (%)",fontsize=7); ax.set_ylim(0,100); ax.grid(alpha=0.3)
    for j in range(i+1,len(axes)): axes[j].set_visible(False)
    fig.suptitle(f"Cumulative explained variance — {_pair_label(pair)} | {DATASET}",fontsize=11)
    fig.tight_layout()
    out=OUT_ROOT/pair/"pca_variance"/"grid_all_layers.png"
    out.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(out,dpi=DPI,bbox_inches="tight")
    plt.show(); print(f"→ {out}")

---
## 11 · Summary — output file tree

In [ ]:
def print_tree(root: Path, indent: int = 0):
    files = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for f in files:
        prefix = "  " * indent
        if f.is_dir():
            print(f"{prefix}📁 {f.name}/")
            print_tree(f, indent + 1)
        else:
            size = f.stat().st_size / 1024
            print(f"{prefix}  {f.name}  ({size:.0f} KB)")

print(f"\n=== Output tree: {OUT_ROOT} ===")
print_tree(OUT_ROOT)